![ecnas-eda](../../assets/ecnas-eda.png)

<a id='top'></a>

### **Title:** EC-NAS Exploratory Data Analysis — Convolutional Cell Energy Consumption
##### Table of Contents

<ul>
    <li><a href='#data-description'><b>1.0 Dataset Technical Description & Relevance</b></a></li>
    <ul style="margin-top: 5px; margin-bottom: 10px;">
        <li style="margin-left: 20px;"><a href='#data-source'>1.1 Dataset Source and Ownership</a></li>
        <li style="margin-left: 20px;"><a href='#overview'>1.2 Dataset Overview & Key Characteristics</a></li>
    </ul>
    <li><a href='#loading'><b>2.0 Data Loading & Initial Inspection</b></a></li>
    <li><a href='#quality'><b>3.0 Data Quality & Missing Values</b></a></li>
    <li><a href='#distributions'><b>4.0 Feature Distributions</b></a></li>
    <li><a href='#target'><b>5.0 Target Variable Analysis (Energy)</b></a></li>
    <li><a href='#relationships'><b>6.0 Feature-Energy Relationships</b></a></li>
    <li><a href='#hardware'><b>7.0 Cross-GPU Hardware Analysis</b></a></li>
    <li><a href='#summary'><b>8.0 EDA Summary & Key Findings</b></a></li>
</ul>

<a id='data-description'></a>

### 1.0 Dataset Technical Description & Relevance

<a id='data-source'></a>

#### 1.1 Dataset Source and Ownership

This project uses [EC-NAS: Energy Consumption Aware Tabular Benchmarks for Neural Architecture Search](https://arxiv.org/abs/2210.06015) (Bakhtiarifard, Igel & Selvan, 2024), published by the Department of Computer Science, University of Copenhagen, with source and data released at [github.com/saintslab/EC-NAS-Bench](https://github.com/saintslab/EC-NAS-Bench) (as cited in the paper).

**Ownership:** EC-NAS was created by the SAINTS lab at the University of Copenhagen, with funding acknowledged under EU Horizon Europe grants No. 101070284 and No. 101070408. It extends the NAS-Bench-101 convolutional-cell search space (Ying et al., 2019) with energy, power, and CO2-equivalent measurements collected via the Carbontracker tool while training cell architectures on CIFAR-10.

The subset of EC-NAS used in this project holds real (non-surrogate) training-run measurements for two search spaces — `4V9E` (91 architectures) and `5V9E` (2,532 architectures) — plus a separate 4-GPU hardware benchmark of the `4V9E` architectures. The paper's headline surrogate model (an MLP trained on this measured subset to predict energy across the full ~423k-architecture space) is out of scope here; only the ground-truth measurements it was trained/validated on are loaded.

<a id='overview'></a>

#### 1.2 Dataset Overview & Key Characteristics

**Measurement Type: Software-Based (Chip-Level Estimate)**

Unlike BUTTER-E's node-level hardware watt-meter (ground truth), EC-NAS measures energy with the Carbontracker tool, which samples GPU/CPU power through vendor APIs (nvidia-smi / Intel RAPL) rather than a physical meter. This is the same category of software-based measurement flagged as a limitation in the BUTTER-E notebook (up to ~40% underestimation vs. ground truth) — a caveat to carry into any feature table that combines the two datasets.

**Scale (locally available subset)**

* 91 architectures × up to 3 repeats (the `4V9E` search space) + 2,532 architectures × 1 repeat (the `5V9E` search space) = 2,805 real training-run records, all trained for a fixed 4-epoch budget on CIFAR-10
* The same 91 `4V9E` architectures additionally re-benchmarked across 4 GPUs × 4 epoch budgets (4/12/36/108) = 1,444 hardware-comparison records
* Reference training hardware for the primary runs: 1× NVIDIA Quadro RTX 6000 GPU + 2× Intel CPU cores (per the paper's Section 2.3, "in-house SLURM cluster")

**Architecture Type: Convolutional Cells (NAS-Bench-101 style) Only**

Every architecture is a directed-acyclic-graph "cell" — a small feedforward block of up to 5 operation nodes (always starting with `input` and ending with `output`, connected by up to 9 edges), stacked into a fixed macro-skeleton and trained on CIFAR-10 image classification. No MLPs are included — BUTTER-E covers that architecture family, across 12 other datasets.

**Files Used in This Project**

| File | Purpose |
|------|---------|
| `graphs/generated_graphs_{4V9E,5V9E}.json` | Per-architecture cell definition: adjacency matrix + per-node operation codes |
| `train_model_results/energy/{4V9E,5V9E}/4_epochs/**/repeat_*/results.json` | Real (non-surrogate) per-run training results: params, time, energy (kWh), CO2eq, GPU/CPU power, accuracy curve |
| `hardware_tfrecords/{quadrortx6000,rtx3060,rtx3090,titanxp}.tfrecord` | The 91 `4V9E` architectures re-benchmarked on 4 GPUs at 4 epoch budgets, in NAS-Bench-101's TFRecord wire format (JSON record wrapping a base64-encoded `ModelMetricsEnergy` protobuf — schema in `src/data/nasbench_proto/model_metrics_energy.proto`) |

**Target Variable**

The primary target is `total_energy_kwh` (from the source field `total_energy (kWh)`), Carbontracker's reported end-to-end training energy in kilowatt-hours. It is converted to joules (`× 3.6e6`) downstream for unit consistency with BUTTER-E's `std_energy`.

In [1]:
# IMPORTS

import json
import glob
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# LOADING DATASETS

# Setting the path to where data is stored
DATA_PATH = "../../data/raw/ec_nas/"

def load_search_space(space):
    """Walk train_model_results/energy/<space>/4_epochs and join each run's
    results.json to its cell definition in graphs/generated_graphs_<space>.json."""
    with open(DATA_PATH + f"graphs/generated_graphs_{space}.json") as f:
        graphs = json.load(f)

    rows = []
    pattern = DATA_PATH + f"train_model_results/energy/{space}/4_epochs/*/*/repeat_*/results.json"
    for path in glob.glob(pattern):
        parts = path.replace(os.sep, "/").split("/")
        graph_hash, repeat = parts[-3], int(parts[-2].split("_")[1])

        with open(path) as f:
            r = json.load(f)

        gpu = r["avg_power_usages:"]["gpu"]
        cpu = r["avg_power_usages:"]["cpu"]
        final_eval = r["evaluation_results"][-1]
        adjacency, ops = graphs[graph_hash]

        rows.append({
            "search_space": space,
            "graph_hash": graph_hash,
            "repeat": repeat,
            "trainable_params": r["trainable_params"],
            "total_time": r["total_time"],
            "total_energy_kwh": r["total_energy (kWh)"],
            "total_co2eq_g": r["total_co2eq (g)"],
            "gpu_device": gpu["devices"][0] if gpu.get("devices") else None,
            "gpu_avg_power_w": gpu["avg_power_usages (W)"][0][0] if gpu.get("avg_power_usages (W)") else None,
            "cpu_avg_power_w": sum(cpu["avg_power_usages (W)"][0]) if cpu.get("avg_power_usages (W)") else None,
            "n_vertices": len(ops),
            "n_edges": sum(sum(row) for row in adjacency),
            "ops": ",".join(str(o) for o in ops),
            "final_epoch": final_eval["epochs"],
            "final_train_acc": final_eval["train_accuracy"],
            "final_val_acc": final_eval["validation_accuracy"],
            "final_test_acc": final_eval["test_accuracy"],
        })
    return pd.DataFrame(rows)

df_4v9e = load_search_space("4V9E")
df_5v9e = load_search_space("5V9E")
df = pd.concat([df_4v9e, df_5v9e], ignore_index=True)

In [3]:
df.head()

,search_space,graph_hash,repeat,trainable_params,total_time,total_energy_kwh,total_co2eq_g,gpu_device,gpu_avg_power_w,cpu_avg_power_w,n_vertices,n_edges,ops,final_epoch,final_train_acc,final_val_acc,final_test_acc
0,4V9E,001a6fcc8b38cbc515be788a86bdc804,1,1756298,154.605136,0.018713,1.646765,NVIDIA Quadro RTX 6000,150.820600,123.300448,4,4,"-1,2,2,-2",4.0,0.284756,0.287059,0.285156
1,4V9E,001a6fcc8b38cbc515be788a86bdc804,2,1756298,121.019199,0.016917,1.640983,NVIDIA Quadro RTX 6000,191.351727,125.230087,4,4,"-1,2,2,-2",4.0,0.366286,0.360677,0.356971
2,4V9E,001a6fcc8b38cbc515be788a86bdc804,3,1756298,119.811754,0.016607,1.461386,Quadro RTX 6000,191.675636,122.224224,4,4,"-1,2,2,-2",4.0,0.403846,0.406550,0.393129
3,4V9E,01f0f1cf81ef9193f2946db78ea759ee,1,5969674,206.902899,0.024870,2.238278,Quadro RTX 6000,156.500368,115.704824,4,5,"-1,0,0,-2",4.0,0.502504,0.499499,0.491486
4,4V9E,01f0f1cf81ef9193f2946db78ea759ee,2,5969674,185.197724,0.024072,2.335026,NVIDIA Quadro RTX 6000,169.575471,124.799288,4,5,"-1,0,0,-2",4.0,0.619291,0.612179,0.606771


In [4]:
df.shape

(2805, 17)

In [5]:
df.dtypes

search_space         object
graph_hash           object
repeat                int64
trainable_params      int64
total_time          float64
total_energy_kwh    float64
total_co2eq_g       float64
gpu_device           object
gpu_avg_power_w     float64
cpu_avg_power_w     float64
n_vertices            int64
n_edges               int64
ops                  object
final_epoch         float64
final_train_acc     float64
final_val_acc       float64
final_test_acc      float64
dtype: object

<a id='quality'></a>

### 3.0 Data Quality & Missing Values

In [6]:
# SEARCH SPACE OVERLAP -- 4V9E vs 5V9E aren't independent samples; resolving
# whether they can be safely concatenated into one feature table

overlap = set(df_4v9e["graph_hash"]) & set(df_5v9e["graph_hash"])
print("4V9E architectures:", df_4v9e["graph_hash"].nunique())
print("5V9E architectures:", df_5v9e["graph_hash"].nunique())
print("architectures shared between the two spaces:", len(overlap))
print("4V9E is a strict subset of 5V9E:", set(df_4v9e["graph_hash"]).issubset(set(df_5v9e["graph_hash"])))

# are the shared architectures' repeat_1 runs duplicated files, or independent repeats?
noise = df[df["graph_hash"].isin(overlap)].groupby("graph_hash")["total_energy_kwh"]
noise_stats = noise.agg(["count", "mean", "std"])
noise_stats["cv"] = noise_stats["std"] / noise_stats["mean"]
print("\nrepeats per shared architecture (3 from 4V9E + 1 from 5V9E):")
print(noise_stats["count"].value_counts())
print("\ntotal_energy_kwh coefficient of variation across repeats of the same architecture:")
print(noise_stats["cv"].describe())

4V9E architectures: 91
5V9E architectures: 2532
architectures shared between the two spaces: 91
4V9E is a strict subset of 5V9E: True

repeats per shared architecture (3 from 4V9E + 1 from 5V9E):
count
4    91
Name: count, dtype: int64

total_energy_kwh coefficient of variation across repeats of the same architecture:
count    91.000000
mean      0.047006
std       0.023513
min       0.007137
25%       0.028661
50%       0.041369
75%       0.064799
max       0.137612
Name: cv, dtype: float64


**Result:** all 91 `4V9E` architectures are also present in `5V9E` with identical cell definitions, and their `5V9E` result is a genuinely independent 4th training repeat (different `total_time`/`total_energy_kwh` from any of the three `4V9E` repeats, not a duplicated file) — run-to-run measurement noise for those 91 shared architectures is small (median CV ≈ 4%, comparably tight to BUTTER-E's repeat-noise check).

**Decision:** for the modeling feature table, use `5V9E` as the primary architecture-level table (each of its 2,532 architectures appears once) rather than the raw `df` concatenation above, which would otherwise let the 91 shared architectures outweigh the other 2,441 by 3-4x. The `4V9E` repeats are kept aside as a noise-characterization set only, not folded into training data.

In [7]:
# NULL CHECKS -- columns that feed feature engineering directly

feature_cols = ["trainable_params", "total_time", "n_vertices", "n_edges", "ops", "gpu_device"]
df[feature_cols].isnull().sum()

trainable_params    0
total_time          0
n_vertices          0
n_edges             0
ops                 0
gpu_device          0
dtype: int64

In [8]:
# VERTEX COUNT & OPERATION VOCABULARY -- "4V9E"/"5V9E" name the max vertex count
# and the fixed 9-edge budget, not a constant graph size; confirm the actual spread
# and decode the numeric op codes against the human-readable labels used elsewhere
# in the dataset (cross-checked against the hardware_tfrecords ops strings in Section 7)

print("n_vertices distribution (all rows):")
print(df["n_vertices"].value_counts().sort_index())

op_vocab = sorted({int(o) for ops in df["ops"] for o in ops.split(",")})
print("\nraw operation codes:", op_vocab)

OP_LABELS = {-1: "input", -2: "output", 0: "conv3x3-bn-relu", 1: "conv1x1-bn-relu", 2: "maxpool3x3"}
print("decoded:", {k: OP_LABELS[k] for k in op_vocab})

n_vertices distribution (all rows):
n_vertices
2       4
3      24
4     336
5    2441
Name: count, dtype: int64

raw operation codes: [-2, -1, 0, 1, 2]
decoded: {-2: 'output', -1: 'input', 0: 'conv3x3-bn-relu', 1: 'conv1x1-bn-relu', 2: 'maxpool3x3'}


In [9]:
# EPOCH BUDGET, DEVICE-NAME CONSISTENCY, AND ENERGY UNITS

print("final_epoch values in the raw per-run files:", df["final_epoch"].unique())
# only a fixed 4-epoch budget was actually trained here; the 12/36/108-epoch
# budgets only exist in the hardware_tfrecords benchmark (Section 7) and, beyond
# that, as surrogate-model predictions over the full ~423k-architecture space
# (paper Fig. 1/2) -- not present locally.

print("\ngpu_device raw string values (same physical GPU, inconsistent label):")
print(df["gpu_device"].value_counts())

print("\ntotal_energy_kwh summary (source unit):")
print(df["total_energy_kwh"].describe())
print("\nconverted to joules (x 3.6e6) for comparability with BUTTER-E's std_energy:")
print((df["total_energy_kwh"] * 3.6e6).describe())

print("\ncorrelation of total_energy_kwh with candidate features:")
print(df[["trainable_params", "total_time", "n_vertices", "n_edges", "total_energy_kwh"]].corr()["total_energy_kwh"])

final_epoch values in the raw per-run files: [4.]

gpu_device raw string values (same physical GPU, inconsistent label):
gpu_device
NVIDIA Quadro RTX 6000    1956
Quadro RTX 6000            849
Name: count, dtype: int64

total_energy_kwh summary (source unit):
count    2805.000000
mean        0.025231
std         0.009088
min         0.006414
25%         0.019090
50%         0.023645
75%         0.029273
max         0.062593
Name: total_energy_kwh, dtype: float64

converted to joules (x 3.6e6) for comparability with BUTTER-E's std_energy:
count      2805.000000
mean      90832.481711
std       32717.648626
min       23090.400000
25%       68724.000000
50%       85122.000000
75%      105382.800000
max      225334.800000
Name: total_energy_kwh, dtype: float64

correlation of total_energy_kwh with candidate features:
trainable_params    0.840650
total_time          0.980611
n_vertices          0.157995
n_edges             0.133589
total_energy_kwh    1.000000
Name: total_energy_kwh, dtype

<a id='hardware'></a>

### 7.0 Cross-GPU Hardware Analysis

In [10]:
# HARDWARE CROSS-GPU BENCHMARK -- the 91 4V9E architectures re-measured on 4 GPUs
# at 4 epoch budgets, stored in NAS-Bench-101's TFRecord wire format: each record's
# payload is a JSON list [hash, epoch_budget, adjacency_str, ops_str, base64-encoded
# ModelMetricsEnergy protobuf] (schema: src/data/nasbench_proto/model_metrics_energy.proto)

import struct
import base64
import sys

sys.path.insert(0, "../..")
from src.data.nasbench_proto import model_metrics_energy_pb2 as pb

HW_PATH = "../../data/raw/ec_nas/hardware_tfrecords/"

def read_tfrecords(path):
    with open(path, "rb") as f:
        data = f.read()
    offset = 0
    while offset < len(data):
        length = struct.unpack("<Q", data[offset:offset + 8])[0]
        offset += 12  # length (8B) + masked length CRC (4B)
        yield data[offset:offset + length]
        offset += length + 4  # data + masked data CRC

hw_rows = []
for gpu_file in ["quadrortx6000", "rtx3060", "rtx3090", "titanxp"]:
    for rec in read_tfrecords(HW_PATH + f"{gpu_file}.tfrecord"):
        graph_hash, epoch_budget, _adjacency, _ops, metrics_b64 = json.loads(rec)
        msg = pb.ModelMetricsEnergy()
        msg.ParseFromString(base64.b64decode(metrics_b64))
        hw_rows.append({
            "gpu_file": gpu_file,
            "graph_hash": graph_hash,
            "epoch_budget": epoch_budget,
            "total_energy_kwh": msg.total_energy,
            "total_time": msg.total_time,
        })

hw_df = pd.DataFrame(hw_rows)
print("hardware benchmark records per GPU:")
print(hw_df["gpu_file"].value_counts())

# does an architecture's energy ranking on one GPU predict its ranking on another?
pivot = hw_df[hw_df["epoch_budget"] == 108].pivot_table(
    index="graph_hash", columns="gpu_file", values="total_energy_kwh"
)
print("\ncross-GPU correlation of total_energy_kwh at the 108-epoch budget:")
print(pivot.corr())

hardware benchmark records per GPU:
gpu_file
quadrortx6000    364
rtx3090          364
titanxp          364
rtx3060          352
Name: count, dtype: int64

cross-GPU correlation of total_energy_kwh at the 108-epoch budget:
gpu_file       quadrortx6000   rtx3060   rtx3090   titanxp
gpu_file                                                  
quadrortx6000       1.000000  0.965583  0.511701  0.981794
rtx3060             0.965583  1.000000  0.712802  0.968688
rtx3090             0.511701  0.712802  1.000000  0.530249
titanxp             0.981794  0.968688  0.530249  1.000000


<a id='summary'></a>

### 8.0 EDA Summary & Key Findings

EDA scoped to what Phase 1 (shared feature table) needs, mirroring `01a_butter_e_eda.ipynb`. No key columns have nulls; the `4V9E`/`5V9E` naming was confirmed to mean *up to* 4 or 5 vertices under a shared 9-edge budget (not a constant cell size), and the 5 numeric operation codes were decoded against the human-readable labels embedded in the hardware benchmark (`input`, `output`, `conv3x3-bn-relu`, `conv1x1-bn-relu`, `maxpool3x3`). One data-quality issue was resolved as a design decision rather than left implicit: `4V9E` is a strict 91-architecture subset of the 2,532-architecture `5V9E` space, so the shared-architecture repeats are quarantined for noise-characterization only (median CV ≈ 4%) — `5V9E` alone is the primary architecture-level table going forward. The target, `total_energy_kwh`, correlates strongly with `total_time` (r≈0.98) and `trainable_params` (r≈0.84) but only weakly with raw vertex/edge counts, so params/time-derived features should carry more of the signal than graph size alone; it will be converted to joules (`× 3.6e6`) to stay unit-consistent with BUTTER-E's `std_energy`. Two limitations are carried forward as documented caveats rather than blockers: EC-NAS measures energy via Carbontracker (software/chip-level sampling), not BUTTER-E's ground-truth node watt-meter, and it covers a single dataset (CIFAR-10) versus BUTTER-E's twelve. The cross-GPU hardware benchmark (Section 7) reproduces the paper's own headline finding on this local subset — energy rankings transfer well between three of the four GPUs (r≈0.97-0.98) but only weakly to the RTX 3090 (r≈0.51-0.71) — so hardware identity needs to stay an explicit feature (or a separate per-hardware model) rather than being assumed away; that benchmark set is reserved for `src/baselines/hardware_scaling.py` rather than folded into the main feature table now. Full distribution plots and outlier visualization are deferred to Section 5 thesis figures — not blocking. Next: `02b_ec_nas_features.ipynb`.

![thanks-eda](../../assets/thanks-eda.png)